# Worker (Colab, respaldo de Kaggle)
Un solo cuaderno para todos los trabajos que el hub deja en Drive (`PoC-Labelling/`): asistente (`asistente/asist_*.zip`),
pre-etiquetado (`lotes/lote_*.zip`) y entrenamiento (`datasets/dataset_*.zip`, solo el mas reciente de cada proyecto).
1. Entorno de ejecucion: **T4 GPU**.
2. Celda 1 una vez por sesion. Celda 2 cada vez que haya trabajos (la UI los muestra en "Pasos externos"). Celda 3: diagnostico.
3. Despues, en el hub: **Reconciliar** (o `python xd.py trabajos --reconciliar --aplicar`) e importar.

El codigo que corre viaja en cada zip (`codigo/worker.py`): este cuaderno solo lo lanza. Contrato: `docs/worker_contrato.md`.
Fuente de verdad: `notebooks/worker.ipynb` del repo (`python xd.py cuadernos`).

In [ ]:
import os, shutil
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/PoC-Labelling'
# versiones fijas = hub (motor/requirements.txt); test_notebooks lo verifica
!pip -q install ultralytics==8.4.174 timm==1.0.30 "clip @ git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40"
if not os.path.exists('/content/sam3.pt'):
    shutil.copy(f'{BASE}/sam3.pt', '/content/sam3.pt')
!nvidia-smi -L
!ls -lh /content/sam3.pt

In [ ]:
import importlib, json, shutil, subprocess, sys, zipfile
from pathlib import Path

BASE = Path('/content/drive/MyDrive/PoC-Labelling')
CARPETAS = [BASE / 'asistente', BASE / 'lotes', BASE / 'datasets']
RUNS = BASE / 'runs'
W = Path('/content/w')
CACHE = globals().get('CACHE', {})          # modelos entre ejecuciones (nunca dos pesados a la vez)


def hecho(zp):
    """Ya procesado: estado.json o _error.txt del worker, o el resultado de antes de P.2.3."""
    s, d = zp.stem, zp.parent
    if (d / f'{s}.estado.json').exists() or (d / f'{s}_error.txt').exists():
        return True
    if s.startswith('asist_'):
        return (d / f'{s}_resultado.json').exists()
    if s.startswith('lote_'):
        return (d / f'{s}_pred.json').exists()
    if s.startswith('dataset_'):
        return any((r / 'weights' / 'metricas.json').exists() for r in RUNS.glob(f'{s}*') if r.name == s or r.name.startswith(s + '_'))
    return True


def cargar_codigo(d):
    """Siempre el codigo que trae el zip."""
    for m in [p.stem for p in (d / 'codigo').glob('*.py')]:
        sys.modules.pop(m, None)
    sys.path[:] = [p for p in sys.path if not p.startswith(str(W))]
    sys.path.insert(0, str(d / 'codigo'))
    return importlib.import_module('worker')


def preparar_asistente():
    """Whisper y Ollama solo si hay trabajos del asistente (una sesion de lotes no los descarga)."""
    subprocess.run("pip -q install faster-whisper==1.2.1 ctranslate2==4.8.4 nvidia-cublas-cu12 "
                   "'nvidia-cudnn-cu12==9.*' nvidia-cuda-runtime-cu12", shell=True, check=True)
    if not shutil.which('ollama'):
        subprocess.run("apt-get -qq install -y zstd lshw > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)


pend = [zp for c in CARPETAS for zp in sorted(c.glob('*.zip')) if not hecho(zp)]
ultimo = {}                                  # entrenamiento: solo el dataset mas reciente de cada proyecto
for zp in (BASE / 'datasets').glob('dataset_p*.zip'):     # entre TODOS: si el ultimo ya se entreno, no se entrena uno viejo
    p = zp.name.split('_')[1]
    ultimo[p] = max(ultimo.get(p, ''), zp.name)
pend = [zp for zp in pend if not zp.name.startswith('dataset_p') or zp.name == ultimo[zp.name.split('_')[1]]]

listos = {}
for zp in pend:
    d = W / zp.stem
    shutil.rmtree(d, ignore_errors=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(d)
    if not (d / 'codigo' / 'worker.py').exists():
        print('SIN worker.py (exportado antes de P.2.3): vuelve a exportarlo ->', zp.name)
        continue
    listos[zp.stem] = (d, zp)
print(f'{len(listos)} trabajo(s) pendiente(s):', ', '.join(listos) or '-')

if listos:
    orden = cargar_codigo(next(iter(listos.values()))[0]).ordenar([(d, s) for s, (d, _) in listos.items()])
    previo, asistente_listo = None, False
    for d, stem in orden:
        worker = cargar_codigo(d)
        g = worker.grupo(d, stem)
        if previo and g != previo:
            worker.liberar(CACHE)            # un solo modelo pesado a la vez
        if g == 'asistente' and not asistente_listo:
            preparar_asistente()
            asistente_listo = True
        worker.procesar(d, listos[stem][1].parent, CACHE, stem=stem, ent=worker.entorno('colab'),
                        runs=RUNS, sam3_pt='/content/sam3.pt', device=0)
        previo = g
print('listo')

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!ollama ps 2>/dev/null || echo 'ollama no instalado (no hubo trabajos del asistente)'
import json
from pathlib import Path
for f in sorted(Path('/content/drive/MyDrive/PoC-Labelling').glob('*/*.estado.json'))[-10:]:
    e = json.loads(f.read_text())
    print(('OK   ' if e['ok'] else 'ERROR'), e['trabajo'], e['seg'], 's', e['resultado'] or e['error'])